In [10]:
import os
from pathlib import Path
import glob
import re

In [11]:
import pyspark.sql.functions as F
import pyspark.sql.types as T
from pyspark.sql import SparkSession

In [12]:
BASE_DIR = "/volume/data/"

DIRECTORY_SILVER = BASE_DIR+'despesas_contabeis/data/silver/'

In [13]:
spark = SparkSession.builder\
    .appName("data_discovery")\
        .config("spark.driver.memory", "8g") \
            .config("spark.executor.memory", "8g").getOrCreate()


In [14]:
df = spark.read.format('parquet').load(DIRECTORY_SILVER+'tb_dados_contabeis_operadoras')

+----+-------+-----------+-----+
|DATA|REG_ANS|cd_contabil|count|
+----+-------+-----------+-----+
|NULL|320251 |358919112  |3    |
|NULL|343315 |216119011  |9    |
|NULL|354511 |211119034  |15   |
|NULL|359033 |121319011  |30   |
|NULL|411248 |458119011  |26   |
|NULL|413275 |216119035  |27   |
|NULL|417416 |462119013  |26   |
|NULL|314668 |463719011  |30   |
|NULL|326755 |411511043  |10   |
|NULL|417556 |461419011  |29   |
|NULL|334561 |465219019  |30   |
|NULL|343889 |253259012  |30   |
|NULL|359033 |218119011  |13   |
|NULL|363766 |461119011  |29   |
|NULL|339954 |123311019  |27   |
|NULL|356417 |411111363  |4    |
|NULL|359033 |311112181  |7    |
|NULL|300713 |123311011  |12   |
|NULL|320251 |463319111  |12   |
|NULL|328596 |411111242  |20   |
|NULL|328596 |411111711  |20   |
|NULL|333875 |133419019  |30   |
|NULL|350141 |123111012  |30   |
|NULL|393321 |461219016  |22   |
|NULL|393321 |411111161  |26   |
|NULL|302490 |463919014  |24   |
|NULL|302490 |311111061  |24   |
|NULL|3146

### Exploratória

In [16]:
#Qtd total de registros

df.count()

8273526

In [21]:
# Avaliar se há duplicadas em relação a conta, reg_ans, e ano_mes_competencia

print(df.columns)
df = df.where(F.col('DATA').isNotNull())

df.groupBy('DATA', 'REG_ANS', 'cd_contabil').count().filter(F.col('count')>1).show(100, truncate=False)



['DATA', 'REG_ANS', 'VL_SALDO_INICIAL', 'VL_SALDO_FINAL', 'Registro_ANS', 'Razao_Social', 'Nome_Fantasia', 'Modalidade', 'UF', 'Data_Registro_ANS', 'Data_Descredenciamento', 'Motivo_do_Descredenciamento', 'n1_cd_contabil', 'n1_ds_conta_contabil', 'n2_cd_contabil', 'n2_ds_conta_contabil', 'n3_cd_contabil', 'n3_ds_conta_contabil', 'n4_cd_contabil', 'n4_ds_conta_contabil', 'n5_cd_contabil', 'n5_ds_conta_contabil', 'n6_cd_contabil', 'n6_ds_conta_contabil', 'n8_cd_contabil', 'n8_ds_conta_contabil', 'cd_contabil', 'ds_conta_contabil']
+----+-------+-----------+-----+
|DATA|REG_ANS|cd_contabil|count|
+----+-------+-----------+-----+
+----+-------+-----------+-----+



In [22]:
df.select('n1_cd_contabil', 'n1_ds_conta_contabil').distinct().show()

+--------------+--------------------+
|n1_cd_contabil|n1_ds_conta_contabil|
+--------------+--------------------+
|             1|               ATIVO|
|             3|            RECEITAS|
|             4|            DESPESAS|
|             7|CONTAS TRANSITÓRI...|
|             6|CONTAS DE DESTINA...|
|             2|             PASSIVO|
+--------------+--------------------+



In [24]:
despesas = df.where(F.col('n1_cd_contabil').startswith('4'))
receitas = df.where(F.col('n1_cd_contabil').startswith('3'))

In [26]:
despesas.show(5)

+----------+-------+----------------+--------------+------------+--------------------+--------------------+--------------------+---+-----------------+----------------------+---------------------------+--------------+--------------------+--------------+--------------------+--------------+--------------------+--------------+--------------------+--------------+--------------------+--------------+--------------------+--------------+--------------------+-----------+--------------------+
|      DATA|REG_ANS|VL_SALDO_INICIAL|VL_SALDO_FINAL|Registro_ANS|        Razao_Social|       Nome_Fantasia|          Modalidade| UF|Data_Registro_ANS|Data_Descredenciamento|Motivo_do_Descredenciamento|n1_cd_contabil|n1_ds_conta_contabil|n2_cd_contabil|n2_ds_conta_contabil|n3_cd_contabil|n3_ds_conta_contabil|n4_cd_contabil|n4_ds_conta_contabil|n5_cd_contabil|n5_ds_conta_contabil|n6_cd_contabil|n6_ds_conta_contabil|n8_cd_contabil|n8_ds_conta_contabil|cd_contabil|   ds_conta_contabil|
+----------+-------+------

In [30]:
despesas.select('n2_cd_contabil', 'n2_ds_conta_contabil').distinct().show(100, truncate=False)

+--------------+-------------------------------------------------+
|n2_cd_contabil|n2_ds_conta_contabil                             |
+--------------+-------------------------------------------------+
|46            |DESPESAS ADMINISTRATIVAS                         |
|41            |EVENTOS INDENIZÁVEIS LÍQUIDOS / SINISTROS RETIDOS|
|45            |DESPESAS FINANCEIRAS                             |
|43            |DESPESAS DE COMERCIALIZAÇÃO                      |
|47            |DESPESAS PATRIMONIAIS                            |
|44            |OUTRAS DESPESAS OPERACIONAIS                     |
+--------------+-------------------------------------------------+



In [37]:
despesas.where(F.col('n2_cd_contabil') == '41').groupBy('DATA', 'Registro_ANS', 'Razao_Social', 'Modalidade').agg(F.sum('VL_SALDO_FINAL').alias('sum_vl_saldo_final')).orderBy(['DATA', 'sum_vl_saldo_final'], ascending=[True, False]).show(20, truncate=False)

+----------+------------+-----------------------------------------------------------+---------------------------------+---------------------+
|DATA      |Registro_ANS|Razao_Social                                               |Modalidade                       |sum_vl_saldo_final   |
+----------+------------+-----------------------------------------------------------+---------------------------------+---------------------+
|2021-10-01|005711      |BRADESCO SAÚDE S.A.                                        |Seguradora Especializada em Saúde|2.557288956086E10    |
|2021-10-01|326305      |AMIL ASSISTÊNCIA MÉDICA INTERNACIONAL S.A.                 |Medicina de Grupo                |1.763307085213E10    |
|2021-10-01|006246      |SUL AMERICA COMPANHIA DE SEGURO SAÚDE                      |Seguradora Especializada em Saúde|1.6409926157320002E10|
|2021-10-01|359017      |NOTRE DAME INTERMÉDICA SAÚDE S.A.                          |Medicina de Grupo                |8.122617135820001E9  |
|2021-

In [39]:
despesas.where(F.col('n2_cd_contabil') == '41').groupBy('DATA', 'Registro_ANS', 'Razao_Social', 'Modalidade').agg(F.sum('VL_SALDO_FINAL').alias('sum_vl_saldo_final')).orderBy(['DATA'], ascending=[True]).show(5, truncate=False)

+----------+------------+-------------------------------------------------------------------+--------------------+--------------------+
|DATA      |Registro_ANS|Razao_Social                                                       |Modalidade          |sum_vl_saldo_final  |
+----------+------------+-------------------------------------------------------------------+--------------------+--------------------+
|2021-10-01|422924      |ODONTOCOOP - OPERADORA DE PLANOS DE SAÚDE LTDA                     |Odontologia de Grupo|6622.07             |
|2021-10-01|359394      |INTEGRAL SERVIÇOS ODONTOLÓGICOS LTDA.                              |Odontologia de Grupo|839559.0            |
|2021-10-01|402834      |UNIMED STA RITA, STA ROSA E SÃO SIMÃO COOP. TRAB. MÉDICO           |Cooperativa Médica  |1.3058591809999997E7|
|2021-10-01|414310      |SAÚDE BRB - CAIXA DE ASSISTÊNCIA                                   |Autogestão          |9.860647016000001E7 |
|2021-10-01|418731      |ASSOCIAÇÃO DE SAÚDE DOS

In [41]:
despesas.select(F.min('DATA').alias('min_data'), F.max('DATA').alias('max_data')).show()

df.select(F.min('DATA').alias('min_data'), F.max('DATA').alias('max_data')).show()

+----------+----------+
|  min_data|  max_data|
+----------+----------+
|2021-10-01|2025-01-01|
+----------+----------+

+----------+----------+
|  min_data|  max_data|
+----------+----------+
|2021-10-01|2025-01-01|
+----------+----------+

